Workspaces > How to Work with All of Us Genomic Data (Hail - Plink)(v8) > Analysis > 01_Get Started with WGS Data_part1_srWGS SNPINDEL

# Importation des données `ancestry_pred`

In [1]:
import os
import subprocess
import pandas as pd

## Accès aux données controlées du `CDR`

Lien vers CDR pour l'accès aux données génétiques -> https://support.researchallofus.org/hc/en-us/articles/29475233432212-Controlled-CDR-Directory

In [7]:
# Définit le chemin vers les fichiers auxiliaires SNP/Indel dans le bucket Google Cloud
auxiliary_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux"
auxiliary_path

'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux'

In [8]:
# Définit le chemin vers le dossier contenant les données d'ascendance
ancestry_path = f'{auxiliary_path}/ancestry'
ancestry_path

'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry'

In [9]:
# Définit le chemin vers le fichier TSV contenant les prédictions d'ascendance
ancestry_pred_path = f'{ancestry_path}/ancestry_preds.tsv'
ancestry_pred_path

'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv'

## Extraction des données via `hail`

`Hail` est une librairie utilisée pour l'analyse génétique à grande échelle.

Nous définissons ici `GRCh38` comme la référence génomique par défaut. Cela permet de s'assurer que toutes les opérations utilisant une référence génomique (par exemple, l'importation de variants ou l'annotation génomique) utiliseront le génome `GRCh38`.

In [ ]:
import hail as hl

# Définit GRCh38 comme référence génomique par défaut pour Hail
hl.default_reference(new_default_reference = "GRCh38")

# Importe un fichier contenant les prédictions d'ascendance, en inférant les types sauf pour les colonnes spécifiées
ancestry_pred = hl.import_table(ancestry_pred_path,
                               key="research_id",
                               impute=True,
                               types={"research_id":"tstr","pca_features":hl.tarray(hl.tfloat)})

## Load 16 PCA and save to bucket

In [12]:
# Calcule le nombre maximal de composantes principales (PCA) présentes dans les prédictions d'ascendance
n_pcs = ancestry_pred.aggregate(hl.agg.max(hl.len(ancestry_pred.pca_features)))

In [ ]:
# Pour chaque composante principale, crée une nouvelle colonne PC1, PC2, ..., avec la valeur correspondante extraite de pca_features
for i in range(n_pcs):
    ancestry_pred = ancestry_pred.annotate(**{f"PC{i+1}": ancestry_pred.pca_features[i]})

In [28]:
# Enlève la clé actuelle sur la table ancestry_pred pour créer une table sans clé
ancestry_pred_unkeyed = ancestry_pred.key_by()

# Prépare la liste des colonnes à exporter : 'research_id' plus toutes les colonnes PC1, PC2, ..., PCn
pc_columns = ['research_id'] + [f"PC{i+1}" for i in range(n_pcs)]

# Sélectionne ces colonnes dans la table sans clé et convertit le résultat en DataFrame pandas
df_pca = ancestry_pred_unkeyed.select(*pc_columns).to_pandas()
df_pca

,research_id,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000000,-0.293561,-0.006345,0.002386,0.001446,0.024304,-0.001529,-0.005621,-0.001204,-0.00092,0.006886,0.004709,0.004915,0.011461,0.002675,-0.001492,0.000896


In [29]:
df_pca.to_csv("ancestry_pcs.tsv", sep="\t", index=False)

# Importation des données extraites 

In [7]:
import os
import subprocess

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))

gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/:
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/ancestry_pcs.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/aou_admixture_estimates_rye_v8.Q
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/pheno_plink.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/relatedness_flagged_samples.tsv

gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/notebooks/:
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/notebooks/1 - Ancestry analysis.ipynb
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/notebooks/2 - Check mixed people.ipynb
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/notebooks/3 - Prepare phenotype - Breast Cancer.ipynb
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/notebooks/4 - GWAS on subset.ipynb
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/notebooks/Tests.ipynb



In [8]:
name_of_file_in_bucket = 'ancestry_pcs.tsv'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
my_dataframe = pd.read_csv(name_of_file_in_bucket, sep='\t')
my_dataframe

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/ancestry_pcs.tsv...
\ [1 files][140.8 MiB/140.8 MiB]                                                
Operation completed over 1 objects/140.8 MiB.                                    


[INFO] ancestry_pcs.tsv is successfully downloaded into your working space


,research_id,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000000,-0.293561,-0.006345,0.002386,0.001446,0.024304,-0.001529,-0.005621,-0.001204,-0.000920,0.006886,0.004709,0.004915,0.011461,0.002675,-0.001492,0.000896
1,1000004,0.101308,0.138703,0.006683,0.053012,0.003345,0.019714,-0.011616,-0.001016,-0.001095,-0.000909,-0.001277,-0.000694,-0.000668,-0.000960,-0.001257,0.000115
2,1000033,0.098486,0.124599,0.009398,0.042617,0.003846,0.026659,-0.018481,-0.001463,-0.001203,0.000465,0.000469,0.000629,0.000019,-0.000210,-0.000013,0.000411
3,1000039,-0.267138,0.004698,0.001046,0.001767,0.031375,0.001713,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
4,1000042,-0.256277,0.004901,-0.002448,0.009514,0.008931,0.010683,0.003820,-0.002210,0.010388,-0.008314,-0.002740,0.003094,0.007370,0.001563,-0.002620,0.006765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414825,9999678,0.098743,0.131769,0.010411,0.048562,0.002988,0.021923,-0.017846,-0.002502,-0.000634,0.000194,0.000252,-0.001290,0.001715,-0.000752,-0.000415,-0.000020
414826,9999697,0.085067,0.028602,-0.107409,0.005666,0.000069,-0.012026,0.010985,0.002458,0.001513,-0.001535,-0.001271,-0.001320,-0.000950,-0.001408,-0.003327,0.000309
414827,9999715,0.099530,0.132836,0.008332,0.050046,0.003166,0.023643,-0.015716,-0.002708,-0.001984,-0.001566,0.000294,0.000889,-0.001221,0.001146,-0.000401,0.000165
414828,9999755,0.064499,0.116637,0.009492,0.042075,-0.005982,-0.022958,0.020151,0.001276,0.000953,-0.000316,-0.003218,0.000905,-0.000271,0.000847,0.001971,-0.000443


# Get admixture with Rye

In [32]:
admixture_estimates_q = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/admixture_estimates/aou_admixture_estimates_rye_v8.Q"
!gsutil -m -u $$GOOGLE_PROJECT cat $admixture_estimates_q | head -n 3

eur	eas	amr	afr	sas	mid	research_id
0.0	0.0026113874691801	0.0	0.940904877573985	0.0	0.0564837349568345	1000000
0.892052409047209	0.0	0.063450014479826	0.0	0.0	0.0444975764729653	1000004


In [33]:
!gsutil -m -u $GOOGLE_PROJECT cp $admixture_estimates_q .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/admixture_estimates/aou_admixture_estimates_rye_v8.Q...
/ [1/1 files][ 32.4 MiB/ 32.4 MiB] 100% Done                                    
Operation completed over 1 objects/32.4 MiB.                                     


In [4]:
import os
import subprocess

destination_filename = 'aou_admixture_estimates_rye_v8.Q'

my_bucket = os.getenv('WORKSPACE_BUCKET')

args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

output.stderr

b'Copying file://./aou_admixture_estimates_rye_v8.Q [Content-Type=application/octet-stream]...\n/ [0 files][    0.0 B/ 32.4 MiB]                                                \r-\r- [0 files][  1.8 MiB/ 32.4 MiB]                                                \r\\\r\\ [0 files][  3.6 MiB/ 32.4 MiB]                                                \r|\r/\r/ [0 files][  5.4 MiB/ 32.4 MiB]                                                \r-\r- [0 files][  7.2 MiB/ 32.4 MiB]                                                \r\\\r|\r| [0 files][  9.0 MiB/ 32.4 MiB]                                                \r/\r/ [0 files][ 10.8 MiB/ 32.4 MiB]                                                \r-\r- [0 files][ 12.6 MiB/ 32.4 MiB]                                                \r\\\r|\r| [0 files][ 14.4 MiB/ 32.4 MiB]                                                \r/\r/ [0 files][ 16.2 MiB/ 32.4 MiB]                                                \r-\r\\\r\\ [0 files][ 18.0 MiB/ 32.4 MiB]    

In [6]:
import pandas as pd

name_of_file_in_bucket = 'aou_admixture_estimates_rye_v8.Q'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
df_rye = pd.read_csv(name_of_file_in_bucket, sep='\t')
df_rye.head()

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/aou_admixture_estimates_rye_v8.Q...
/ [1 files][ 32.4 MiB/ 32.4 MiB]                                                
Operation completed over 1 objects/32.4 MiB.                                     


[INFO] aou_admixture_estimates_rye_v8.Q is successfully downloaded into your working space


,eur,eas,amr,afr,sas,mid,research_id
0,0.000000,0.002611,0.000000,0.940905,0.000000,0.056484,1000000
1,0.892052,0.000000,0.063450,0.000000,0.000000,0.044498,1000004
2,0.886335,0.000000,0.039803,0.000000,0.073862,0.000000,1000033
3,0.045004,0.003594,0.000000,0.874350,0.003528,0.073524,1000039
4,0.115799,0.014158,0.000000,0.853614,0.000000,0.016430,1000042


In [31]:
df_rye.head(30)

,eur,eas,amr,afr,sas,mid,research_id
0,0.000000,0.002611,0.000000,0.940905,0.000000,0.056484,1000000
1,0.892052,0.000000,0.063450,0.000000,0.000000,0.044498,1000004
2,0.886335,0.000000,0.039803,0.000000,0.073862,0.000000,1000033
3,0.045004,0.003594,0.000000,0.874350,0.003528,0.073524,1000039
4,0.115799,0.014158,0.000000,0.853614,0.000000,0.016430,1000042
5,0.000000,0.939365,0.000000,0.007806,0.031606,0.021223,1000045
6,0.781992,0.000000,0.120046,0.000000,0.028276,0.069686,1000059
7,0.906560,0.000000,0.019062,0.000000,0.046401,0.027977,1000061
8,0.783972,0.000952,0.091612,0.000000,0.042836,0.080627,1000062
9,0.894111,0.000000,0.050451,0.000000,0.041453,0.013986,1000070


# Relatedness

In [3]:
relatedness = f'{auxiliary_path}/relatedness'
relatedness

'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness'

In [4]:
related_samples_path = f'{relatedness}/relatedness_flagged_samples.tsv'

In [9]:
!gsutil -u $$GOOGLE_PROJECT cp $related_samples_path .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv...
/ [1 files][239.0 KiB/239.0 KiB]                                                
Operation completed over 1 objects/239.0 KiB.                                    


In [2]:
destination_filename = 'relatedness_flagged_samples.tsv'

my_bucket = os.getenv('WORKSPACE_BUCKET')

args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

output.stderr

b'CommandException: No URLs matched: ./relatedness_flagged_samples.tsv\n'

In [3]:
name_of_file_in_bucket = 'relatedness_flagged_samples.tsv'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
relatedness = pd.read_csv(name_of_file_in_bucket, sep='\t')
relatedness.head(30)

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/relatedness_flagged_samples.tsv...
/ [1 files][239.0 KiB/239.0 KiB]                                                
Operation completed over 1 objects/239.0 KiB.                                    


[INFO] relatedness_flagged_samples.tsv is successfully downloaded into your working space


,sample_id
0,1000039
1,1000091
2,1000109
3,1000127
4,1000320
5,1000452
6,1000530
7,1000612
8,1000700
9,1000774


In [4]:
relatedness

,sample_id
0,1000039
1,1000091
2,1000109
3,1000127
4,1000320
...,...
30579,9996831
30580,9996887
30581,9997301
30582,9998505
